# 🎵 Reto 10: Analizador de Plataforma de Streaming Musical

## Instituto Politécnico Nacional
### Programación para Ciencia de Datos

---

## 📋 Contexto del Reto

¡Felicidades! Has sido contratado como **Data Analyst** en **"SoundWave"**, una plataforma de streaming musical que compite con Spotify y Apple Music. Tu jefe te ha pedido analizar los datos de la plataforma para entender mejor el comportamiento de los usuarios y el rendimiento de los artistas.

```
    ╔══════════════════════════════════════════════════════════════╗
    ║                                                              ║
    ║   🎧 SOUNDWAVE ANALYTICS DASHBOARD 🎧                        ║
    ║                                                              ║
    ║   ┌─────────────┐  ┌─────────────┐  ┌─────────────┐         ║
    ║   │  USUARIOS   │  │  CANCIONES  │  │   STREAMS   │         ║
    ║   │   50,000+   │  │   100,000+  │  │  10M+ /día  │         ║
    ║   └─────────────┘  └─────────────┘  └─────────────┘         ║
    ║                                                              ║
    ║   Tu misión: Analizar y combinar datos para generar         ║
    ║   insights que impulsen el crecimiento de la plataforma     ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
```

---

## 🎯 Objetivos de Aprendizaje

Al completar este reto, dominarás:

| Habilidad | Aplicación en el Reto |
|-----------|----------------------|
| `pd.concat()` | Combinar datos de múltiples meses |
| `pd.merge()` | Unir información de artistas, canciones y usuarios |
| `groupby()` | Calcular estadísticas por artista, género y país |
| `pivot_table()` | Crear reportes ejecutivos de streaming |
| `melt()` | Transformar datos para visualización |

---

## 📊 Datos Disponibles

Trabajarás con 5 conjuntos de datos que representan diferentes aspectos de la plataforma:

```
    ┌──────────────────────────────────────────────────────────────┐
    │                    MODELO DE DATOS                           │
    │                                                              │
    │   ┌─────────────┐         ┌─────────────┐                   │
    │   │  artistas   │────────▶│  canciones  │                   │
    │   │             │         │             │                   │
    │   │ artista_id  │         │ cancion_id  │                   │
    │   │ nombre      │         │ artista_id  │                   │
    │   │ pais_origen │         │ titulo      │                   │
    │   │ genero      │         │ duracion_s  │                   │
    │   └─────────────┘         │ fecha_pub   │                   │
    │                           └──────┬──────┘                   │
    │                                  │                          │
    │                                  ▼                          │
    │   ┌─────────────┐         ┌─────────────┐                   │
    │   │  usuarios   │────────▶│   streams   │                   │
    │   │             │         │             │                   │
    │   │ usuario_id  │         │ stream_id   │                   │
    │   │ nombre      │         │ usuario_id  │                   │
    │   │ pais        │         │ cancion_id  │                   │
    │   │ tipo_cuenta │         │ fecha       │                   │
    │   │ fecha_reg   │         │ completo    │                   │
    │   └─────────────┘         └─────────────┘                   │
    │                                                              │
    └──────────────────────────────────────────────────────────────┘
```

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np

# Configuración para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
# ═══════════════════════════════════════════════════════════════
# DATOS DE LA PLATAFORMA SOUNDWAVE
# ═══════════════════════════════════════════════════════════════

# 1. CATÁLOGO DE ARTISTAS
artistas = pd.DataFrame({
    'artista_id': ['A001', 'A002', 'A003', 'A004', 'A005', 'A006', 'A007', 'A008'],
    'nombre': ['Bad Bunny', 'Taylor Swift', 'BTS', 'Peso Pluma', 'Dua Lipa', 
               'Feid', 'The Weeknd', 'Karol G'],
    'pais_origen': ['Puerto Rico', 'USA', 'Corea del Sur', 'México', 'Reino Unido',
                   'Colombia', 'Canadá', 'Colombia'],
    'genero_principal': ['Reggaeton', 'Pop', 'K-Pop', 'Regional Mexicano', 'Pop',
                        'Reggaeton', 'R&B', 'Reggaeton'],
    'seguidores_millones': [45.2, 52.1, 48.7, 12.3, 38.5, 18.9, 41.2, 35.6]
})

# 2. CATÁLOGO DE CANCIONES
canciones = pd.DataFrame({
    'cancion_id': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008',
                   'C009', 'C010', 'C011', 'C012', 'C013', 'C014', 'C015', 'C016'],
    'artista_id': ['A001', 'A001', 'A002', 'A002', 'A003', 'A003', 'A004', 'A004',
                   'A005', 'A005', 'A006', 'A006', 'A007', 'A007', 'A008', 'A008'],
    'titulo': ['Monaco', 'Tití Me Preguntó', 'Anti-Hero', 'Shake It Off', 
               'Dynamite', 'Butter', 'Ella Baila Sola', 'Lady Gaga',
               'Levitating', 'Dance The Night', 'Ferxxo 100', 'Normal',
               'Blinding Lights', 'Save Your Tears', 'TQG', 'Provenza'],
    'duracion_segundos': [198, 243, 201, 219, 199, 165, 232, 185,
                          203, 176, 245, 198, 200, 215, 197, 208],
    'fecha_publicacion': ['2024-01-15', '2022-05-06', '2022-10-21', '2014-08-18',
                          '2020-08-21', '2021-05-21', '2023-03-10', '2023-06-23',
                          '2020-10-01', '2023-05-25', '2022-03-25', '2023-08-15',
                          '2020-03-20', '2021-02-05', '2023-02-24', '2022-04-22'],
    'explicito': [True, True, False, False, False, False, True, True,
                  False, False, True, True, False, False, True, False]
})

# 3. USUARIOS DE LA PLATAFORMA
usuarios = pd.DataFrame({
    'usuario_id': ['U001', 'U002', 'U003', 'U004', 'U005', 'U006', 'U007', 'U008',
                   'U009', 'U010', 'U011', 'U012', 'U013', 'U014', 'U015', 'U016'],
    'nombre': ['Carlos', 'María', 'Juan', 'Ana', 'Pedro', 'Sofía', 'Diego', 'Valentina',
               'Miguel', 'Camila', 'Andrés', 'Isabella', 'Ricardo', 'Fernanda', 'Jorge', 'Lucía'],
    'pais': ['México', 'México', 'Colombia', 'Colombia', 'España', 'España', 
             'Argentina', 'Argentina', 'Chile', 'Chile', 'Perú', 'Perú',
             'México', 'México', 'Colombia', 'Colombia'],
    'tipo_cuenta': ['Premium', 'Free', 'Premium', 'Premium', 'Free', 'Premium',
                    'Premium', 'Free', 'Premium', 'Free', 'Free', 'Premium',
                    'Premium', 'Free', 'Premium', 'Premium'],
    'edad': [22, 25, 19, 31, 28, 24, 35, 20, 27, 23, 29, 18, 33, 21, 26, 30],
    'fecha_registro': ['2023-01-15', '2023-03-22', '2022-11-08', '2023-06-17',
                       '2022-08-30', '2023-02-14', '2021-12-01', '2023-09-05',
                       '2022-05-20', '2023-07-11', '2023-04-03', '2022-10-25',
                       '2021-06-15', '2023-08-19', '2022-02-28', '2023-01-07']
})

# 4. STREAMS DE ENERO 2024
streams_enero = pd.DataFrame({
    'stream_id': [f'S{i:04d}' for i in range(1, 26)],
    'usuario_id': ['U001', 'U002', 'U001', 'U003', 'U004', 'U005', 'U006', 'U007',
                   'U008', 'U009', 'U010', 'U011', 'U012', 'U001', 'U002', 'U003',
                   'U013', 'U014', 'U015', 'U016', 'U004', 'U005', 'U006', 'U007', 'U008'],
    'cancion_id': ['C001', 'C003', 'C007', 'C005', 'C001', 'C009', 'C013', 'C015',
                   'C002', 'C011', 'C004', 'C006', 'C008', 'C010', 'C012', 'C014',
                   'C001', 'C003', 'C007', 'C015', 'C016', 'C002', 'C005', 'C009', 'C011'],
    'fecha': ['2024-01-05'] * 8 + ['2024-01-12'] * 8 + ['2024-01-20'] * 9,
    'escucha_completa': [True, True, False, True, True, True, True, False,
                         True, True, True, False, True, True, False, True,
                         True, True, True, True, False, True, True, True, False],
    'mes': ['Enero'] * 25
})

# 5. STREAMS DE FEBRERO 2024
streams_febrero = pd.DataFrame({
    'stream_id': [f'S{i:04d}' for i in range(26, 51)],
    'usuario_id': ['U002', 'U004', 'U006', 'U008', 'U010', 'U012', 'U014', 'U016',
                   'U001', 'U003', 'U005', 'U007', 'U009', 'U011', 'U013', 'U015',
                   'U002', 'U004', 'U006', 'U008', 'U010', 'U012', 'U014', 'U016', 'U001'],
    'cancion_id': ['C001', 'C001', 'C001', 'C003', 'C003', 'C005', 'C007', 'C007',
                   'C009', 'C011', 'C013', 'C015', 'C002', 'C004', 'C006', 'C008',
                   'C010', 'C012', 'C014', 'C016', 'C001', 'C007', 'C015', 'C003', 'C013'],
    'fecha': ['2024-02-03'] * 8 + ['2024-02-14'] * 8 + ['2024-02-25'] * 9,
    'escucha_completa': [True, True, True, False, True, True, True, True,
                         True, False, True, True, True, True, False, True,
                         True, True, True, False, True, True, True, True, True],
    'mes': ['Febrero'] * 25
})

print("✅ Datos cargados exitosamente")
print(f"\n📊 Resumen de datos disponibles:")
print(f"   • Artistas: {len(artistas)} registros")
print(f"   • Canciones: {len(canciones)} registros")
print(f"   • Usuarios: {len(usuarios)} registros")
print(f"   • Streams Enero: {len(streams_enero)} registros")
print(f"   • Streams Febrero: {len(streams_febrero)} registros")

✅ Datos cargados exitosamente

📊 Resumen de datos disponibles:
   • Artistas: 8 registros
   • Canciones: 16 registros
   • Usuarios: 16 registros
   • Streams Enero: 25 registros
   • Streams Febrero: 25 registros


---

## 🔥 Parte 1: Combinando Datos Históricos (concat)

### Ejercicio 1.1: Unificar Streams Mensuales

El equipo de analytics necesita un **dataset unificado** con todos los streams de Enero y Febrero para hacer análisis históricos.

**Tu tarea:**
1. Combinar `streams_enero` y `streams_febrero` en un solo DataFrame llamado `streams_total`
2. Reiniciar el índice para que sea consecutivo
3. Mostrar el número total de streams y las primeras 5 filas

```
    ANTES:                              DESPUÉS:
    
    streams_enero (25 filas)            streams_total (50 filas)
    ┌───────────────────────┐           ┌───────────────────────┐
    │ Enero streams...      │     +     │ Enero + Febrero       │
    └───────────────────────┘           │ combinados            │
    streams_febrero (25 filas)    ═══▶  │ índice reiniciado     │
    ┌───────────────────────┐           └───────────────────────┘
    │ Febrero streams...    │
    └───────────────────────┘
```

In [3]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 1.1: Combinar streams de Enero y Febrero            ║
# ╚═══════════════════════════════════════════════════════════════╝

# 1. Combinar los streams de enero y febrero.
# 2. ignore_index=True reinicia el índice automáticamente.
streams_total = pd.concat([streams_enero, streams_febrero], ignore_index=True)

# Verificación
print(f"Total de streams combinados: {len(streams_total)}")
print(streams_total.head())


Total de streams combinados: 50
  stream_id usuario_id cancion_id       fecha  escucha_completa    mes
0     S0001       U001       C001  2024-01-05              True  Enero
1     S0002       U002       C003  2024-01-05              True  Enero
2     S0003       U001       C007  2024-01-05             False  Enero
3     S0004       U003       C005  2024-01-05              True  Enero
4     S0005       U004       C001  2024-01-05              True  Enero


---

## 🔗 Parte 2: Enriqueciendo Datos (merge)

### Ejercicio 2.1: Streams con Información de Canciones

Para el reporte semanal, necesitamos saber **qué canciones** se escucharon en cada stream.

**Tu tarea:**
1. Unir `streams_total` con `canciones` usando `cancion_id` como llave
2. El resultado debe llamarse `streams_canciones`
3. Usa un `inner join` (merge por defecto)

```
    streams_total              canciones
    ┌─────────────────┐       ┌─────────────────┐
    │ stream_id       │       │ cancion_id      │
    │ usuario_id      │       │ artista_id      │
    │ cancion_id  ────┼──────▶│ titulo          │
    │ fecha           │       │ duracion_seg    │
    │ escucha_completa│       │ fecha_pub       │
    └─────────────────┘       └─────────────────┘
```

In [4]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 2.1: Unir streams con información de canciones      ║
# ╚═══════════════════════════════════════════════════════════════╝

# Unimos streams_total con canciones usando la llave cancion_id.
streams_canciones = pd.merge(
    streams_total,
    canciones,
    on='cancion_id',
    how='inner'
)

# Verificación
print(streams_canciones[['stream_id', 'titulo', 'fecha', 'escucha_completa']].head(10))


  stream_id            titulo       fecha  escucha_completa
0     S0001            Monaco  2024-01-05              True
1     S0002         Anti-Hero  2024-01-05              True
2     S0003   Ella Baila Sola  2024-01-05             False
3     S0004          Dynamite  2024-01-05              True
4     S0005            Monaco  2024-01-05              True
5     S0006        Levitating  2024-01-05              True
6     S0007   Blinding Lights  2024-01-05              True
7     S0008               TQG  2024-01-05             False
8     S0009  Tití Me Preguntó  2024-01-12              True
9     S0010        Ferxxo 100  2024-01-12              True


### Ejercicio 2.2: Información Completa de Streams

Ahora el equipo de producto quiere un **dataset completo** que incluya información del artista.

**Tu tarea:**
1. Partiendo de `streams_canciones`, unir con `artistas` usando `artista_id`
2. Guardar resultado en `streams_completos`
3. Seleccionar solo las columnas: `stream_id`, `titulo`, `nombre` (del artista), `genero_principal`, `fecha`, `escucha_completa`

In [5]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 2.2: Agregar información de artistas                ║
# ╚═══════════════════════════════════════════════════════════════╝

# Agregamos la información del artista usando artista_id.
streams_completos = pd.merge(
    streams_canciones,
    artistas,
    on='artista_id',
    how='inner'
)

# Selecciona las columnas relevantes.
columnas_reporte = ['stream_id', 'titulo', 'nombre', 'genero_principal', 'fecha', 'escucha_completa']
streams_reporte = streams_completos[columnas_reporte]

# Verificación
print(streams_reporte.head(10))


  stream_id            titulo        nombre   genero_principal       fecha  \
0     S0001            Monaco     Bad Bunny          Reggaeton  2024-01-05   
1     S0002         Anti-Hero  Taylor Swift                Pop  2024-01-05   
2     S0003   Ella Baila Sola    Peso Pluma  Regional Mexicano  2024-01-05   
3     S0004          Dynamite           BTS              K-Pop  2024-01-05   
4     S0005            Monaco     Bad Bunny          Reggaeton  2024-01-05   
5     S0006        Levitating      Dua Lipa                Pop  2024-01-05   
6     S0007   Blinding Lights    The Weeknd                R&B  2024-01-05   
7     S0008               TQG       Karol G          Reggaeton  2024-01-05   
8     S0009  Tití Me Preguntó     Bad Bunny          Reggaeton  2024-01-12   
9     S0010        Ferxxo 100          Feid          Reggaeton  2024-01-12   

   escucha_completa  
0              True  
1              True  
2             False  
3              True  
4              True  
5        

### Ejercicio 2.3: Análisis de Usuarios Premium vs Free

El equipo comercial quiere comparar el comportamiento de usuarios **Premium vs Free**.

**Tu tarea:**
1. Unir `streams_total` con `usuarios` usando `usuario_id`
2. Calcular el porcentaje de escuchas completas por tipo de cuenta
3. ¿Qué tipo de usuario escucha más canciones completas?

```
    ┌─────────────────────────────────────────┐
    │     PREGUNTA DE NEGOCIO:                │
    │                                         │
    │     ¿Los usuarios Premium escuchan      │
    │     más canciones completas que los     │
    │     usuarios Free?                      │
    │                                         │
    │     💎 Premium: ___% completas          │
    │     🆓 Free: ___% completas             │
    └─────────────────────────────────────────┘
```

In [6]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 2.3: Comparar comportamiento Premium vs Free        ║
# ╚═══════════════════════════════════════════════════════════════╝

# 1. Unimos streams_total con usuarios.
streams_usuarios = pd.merge(
    streams_total,
    usuarios,
    on='usuario_id',
    how='inner'
)

# 2. Calculamos el promedio de escucha_completa por tipo de cuenta.
# Como True=1 y False=0, el promedio equivale al porcentaje de canciones completas.
completion_por_cuenta = (
    streams_usuarios
    .groupby('tipo_cuenta')['escucha_completa']
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

# 3. Identificamos quién tiene mayor porcentaje.
tipo_mayor_completion = completion_por_cuenta.idxmax()

print("Porcentaje de escuchas completas por tipo de cuenta:")
print(completion_por_cuenta)
print(f"\nEl tipo de usuario con más canciones completas es: {tipo_mayor_completion}")


Porcentaje de escuchas completas por tipo de cuenta:
tipo_cuenta
Premium    83.87
Free       73.68
Name: escucha_completa, dtype: float64

El tipo de usuario con más canciones completas es: Premium


---

## 📈 Parte 3: Análisis Agregados (groupby)

### Ejercicio 3.1: Top Artistas por Streams

El equipo de contenido quiere saber cuáles son los **artistas más escuchados**.

**Tu tarea:**
1. Usando `streams_completos`, agrupa por nombre de artista
2. Cuenta el número total de streams por artista
3. Ordena de mayor a menor
4. Muestra el top 5

```
    🏆 TOP 5 ARTISTAS POR STREAMS
    ╔═══════════════════════════════════════╗
    ║  #1  │  ???????????  │  ?? streams   ║
    ║  #2  │  ???????????  │  ?? streams   ║
    ║  #3  │  ???????????  │  ?? streams   ║
    ║  #4  │  ???????????  │  ?? streams   ║
    ║  #5  │  ???????????  │  ?? streams   ║
    ╚═══════════════════════════════════════╝
```

In [7]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 3.1: Top 5 artistas más escuchados                  ║
# ╚═══════════════════════════════════════════════════════════════╝

top_artistas = (
    streams_completos
    .groupby('nombre')
    .size()
    .sort_values(ascending=False)
    .head(5)
)

print("🏆 TOP 5 ARTISTAS POR STREAMS")
print(top_artistas)


🏆 TOP 5 ARTISTAS POR STREAMS
nombre
Bad Bunny       10
Peso Pluma       7
Taylor Swift     7
Karol G          6
Feid             5
dtype: int64


### Ejercicio 3.2: Estadísticas por Género Musical

El equipo de playlists quiere entender qué géneros tienen mejor **engagement**.

**Tu tarea:**
1. Agrupa por `genero_principal`
2. Calcula:
   - Total de streams
   - Promedio de escuchas completas (tasa de completion)
   - Número de artistas únicos
3. Ordena por total de streams

💡 **Hint:** Usa `.agg()` con un diccionario para calcular múltiples estadísticas

In [8]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 3.2: Estadísticas por género musical                ║
# ╚═══════════════════════════════════════════════════════════════╝

estadisticas_genero = (
    streams_completos
    .groupby('genero_principal')
    .agg(
        total_streams=('stream_id', 'count'),
        tasa_completion=('escucha_completa', 'mean'),
        artistas_unicos=('nombre', 'nunique')
    )
    .sort_values('total_streams', ascending=False)
)

# Convertimos la tasa a porcentaje para que sea más fácil de leer.
estadisticas_genero['tasa_completion'] = (estadisticas_genero['tasa_completion'] * 100).round(2)

print(estadisticas_genero)


                   total_streams  tasa_completion  artistas_unicos
genero_principal                                                  
Reggaeton                     21            71.43                3
Pop                           12            91.67                2
Regional Mexicano              7            85.71                1
K-Pop                          5            60.00                1
R&B                            5           100.00                1


### Ejercicio 3.3: Análisis por País y Mes

El equipo de expansión internacional quiere ver la **evolución del streaming por país**.

**Tu tarea:**
1. Primero, une `streams_total` con `usuarios` para obtener el país del usuario
2. Agrupa por `pais` y `mes`
3. Cuenta el número de streams
4. Muestra qué país creció más entre Enero y Febrero

In [9]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 3.3: Streams por país y mes                         ║
# ╚═══════════════════════════════════════════════════════════════╝

# 1. Ya tenemos streams_usuarios de la parte 2.3.
#    Si no existiera, lo volvemos a crear para que esta celda sea segura.
streams_usuarios = pd.merge(
    streams_total,
    usuarios,
    on='usuario_id',
    how='inner'
)

# 2. Agrupar por país y mes.
streams_pais_mes = (
    streams_usuarios
    .groupby(['pais', 'mes'])
    .size()
    .reset_index(name='streams')
)

print("Streams por país y mes:")
print(streams_pais_mes)

# 3. Calcular crecimiento de Febrero contra Enero.
crecimiento_pais = (
    streams_pais_mes
    .pivot(index='pais', columns='mes', values='streams')
    .fillna(0)
)

crecimiento_pais['crecimiento'] = crecimiento_pais.get('Febrero', 0) - crecimiento_pais.get('Enero', 0)
crecimiento_pais = crecimiento_pais.sort_values('crecimiento', ascending=False)

pais_mayor_crecimiento = crecimiento_pais['crecimiento'].idxmax()
mayor_crecimiento = int(crecimiento_pais.loc[pais_mayor_crecimiento, 'crecimiento'])

print("\nCrecimiento de Enero a Febrero:")
print(crecimiento_pais)
print(f"\nPaís con mayor crecimiento: {pais_mayor_crecimiento} ({mayor_crecimiento:+d} streams)")


Streams por país y mes:
         pais      mes  streams
0   Argentina    Enero        4
1   Argentina  Febrero        3
2       Chile    Enero        2
3       Chile  Febrero        3
4    Colombia    Enero        6
5    Colombia  Febrero        6
6      España    Enero        4
7      España  Febrero        3
8      México    Enero        7
9      México  Febrero        7
10       Perú    Enero        2
11       Perú  Febrero        3

Crecimiento de Enero a Febrero:
mes        Enero  Febrero  crecimiento
pais                                  
Chile          2        3            1
Perú           2        3            1
México         7        7            0
Colombia       6        6            0
Argentina      4        3           -1
España         4        3           -1

País con mayor crecimiento: Chile (+1 streams)


---

## 📊 Parte 4: Reportes Ejecutivos (pivot_table)

### Ejercicio 4.1: Matriz de Streams por Género y País

El CEO quiere un **reporte visual** que muestre streams por género y país en formato de matriz.

**Tu tarea:**
1. Usa los datos combinados de streams con usuarios y canciones/artistas
2. Crea una pivot_table donde:
   - Filas: `genero_principal`
   - Columnas: `pais` (del usuario)
   - Valores: conteo de streams
3. Rellena los valores faltantes con 0

```
    📊 MATRIZ DE STREAMS
    
                    │ México │ Colombia │ España │ Argentina │ ...
    ────────────────┼────────┼──────────┼────────┼───────────┼────
    Reggaeton       │   ??   │    ??    │   ??   │    ??     │
    Pop             │   ??   │    ??    │   ??   │    ??     │
    K-Pop           │   ??   │    ??    │   ??   │    ??     │
    Regional Mex    │   ??   │    ??    │   ??   │    ??     │
    R&B             │   ??   │    ??    │   ??   │    ??     │
```

In [10]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 4.1: Pivot table de streams por género y país       ║
# ╚═══════════════════════════════════════════════════════════════╝

# DataFrame con género musical y país del usuario.
streams_genero_pais = (
    streams_completos
    .merge(
        usuarios[['usuario_id', 'pais', 'tipo_cuenta']],
        on='usuario_id',
        how='inner'
    )
)

# Matriz de streams por género y país.
pivot_genero_pais = pd.pivot_table(
    streams_genero_pais,
    index='genero_principal',
    columns='pais',
    values='stream_id',
    aggfunc='count',
    fill_value=0
)

print("📊 MATRIZ DE STREAMS POR GÉNERO Y PAÍS")
print(pivot_genero_pais)


📊 MATRIZ DE STREAMS POR GÉNERO Y PAÍS
pais               Argentina  Chile  Colombia  España  México  Perú
genero_principal                                                   
K-Pop                      0      0         1       1       1     2
Pop                        2      2         1       1       5     1
R&B                        0      0         1       3       1     0
Reggaeton                  5      3         6       2       5     0
Regional Mexicano          0      0         3       0       2     2


### Ejercicio 4.2: Reporte de Engagement por Mes y Tipo de Cuenta

El equipo de producto quiere ver cómo varía el **engagement** (escuchas completas) entre usuarios Premium y Free a lo largo de los meses.

**Tu tarea:**
1. Crea una pivot_table donde:
   - Filas: `mes`
   - Columnas: `tipo_cuenta`
   - Valores: promedio de `escucha_completa`
2. Interpreta: ¿Mejoró el engagement en Febrero?

```
    📈 TASA DE COMPLETION POR MES Y TIPO DE CUENTA
    
              │  Premium  │   Free   │
    ──────────┼───────────┼──────────│
    Enero     │   ??%     │   ??%    │
    Febrero   │   ??%     │   ??%    │
```

In [11]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 4.2: Engagement por mes y tipo de cuenta            ║
# ╚═══════════════════════════════════════════════════════════════╝

# Pivot table con promedio de escucha_completa por mes y tipo de cuenta.
pivot_engagement = pd.pivot_table(
    streams_usuarios,
    index='mes',
    columns='tipo_cuenta',
    values='escucha_completa',
    aggfunc='mean',
    fill_value=0
)

# Lo convertimos a porcentaje.
pivot_engagement_pct = (pivot_engagement * 100).round(2)

print("📈 TASA DE COMPLETION POR MES Y TIPO DE CUENTA")
print(pivot_engagement_pct)

# Interpretación.
promedio_mensual = pivot_engagement.mean(axis=1)
mejora_febrero = promedio_mensual.get('Febrero', 0) > promedio_mensual.get('Enero', 0)

if mejora_febrero:
    print("\nSí, el engagement mejoró en Febrero.")
else:
    print("\nNo, el engagement no mejoró en Febrero.")


📈 TASA DE COMPLETION POR MES Y TIPO DE CUENTA
tipo_cuenta   Free  Premium
mes                        
Enero        66.67    81.25
Febrero      80.00    86.67

Sí, el engagement mejoró en Febrero.


---

## 🔄 Parte 5: Transformación de Datos (melt) - BONUS

### Ejercicio 5.1: Preparar Datos para Visualización

El equipo de BI necesita los datos en **formato largo** para crear gráficos en su herramienta de visualización.

**Tu tarea:**
1. Toma la pivot_table del ejercicio 4.1 (género vs país)
2. Usa `melt()` para convertirla a formato largo
3. El resultado debe tener columnas: `genero_principal`, `pais`, `streams`

```
    FORMATO ANCHO (pivot):              FORMATO LARGO (melt):
    
            │ MX │ CO │ ES │            genero    │ pais │ streams
    ────────┼────┼────┼────│            ──────────┼──────┼────────
    Reggaeton│ 5  │ 3  │ 2 │    ═══▶   Reggaeton │  MX  │   5
    Pop     │ 4  │ 2  │ 3 │            Reggaeton │  CO  │   3
                                        Reggaeton │  ES  │   2
                                        Pop       │  MX  │   4
                                        ...       │ ...  │  ...
```

In [12]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 5.1 (BONUS): Convertir pivot a formato largo        ║
# ╚═══════════════════════════════════════════════════════════════╝

# Reiniciamos el índice para que genero_principal vuelva a ser columna.
pivot_genero_pais_reset = pivot_genero_pais.reset_index()

# Convertimos de formato ancho a formato largo.
streams_formato_largo = pd.melt(
    pivot_genero_pais_reset,
    id_vars=['genero_principal'],
    var_name='pais',
    value_name='streams'
)

print("Formato largo para visualización:")
print(streams_formato_largo)


Formato largo para visualización:
     genero_principal       pais  streams
0               K-Pop  Argentina        0
1                 Pop  Argentina        2
2                 R&B  Argentina        0
3           Reggaeton  Argentina        5
4   Regional Mexicano  Argentina        0
5               K-Pop      Chile        0
6                 Pop      Chile        2
7                 R&B      Chile        0
8           Reggaeton      Chile        3
9   Regional Mexicano      Chile        0
10              K-Pop   Colombia        1
11                Pop   Colombia        1
12                R&B   Colombia        1
13          Reggaeton   Colombia        6
14  Regional Mexicano   Colombia        3
15              K-Pop     España        1
16                Pop     España        1
17                R&B     España        3
18          Reggaeton     España        2
19  Regional Mexicano     España        0
20              K-Pop     México        1
21                Pop     México        5


---

## 🎯 Desafío Final: Reporte Ejecutivo Completo

El CEO de SoundWave necesita un **reporte ejecutivo** para la junta de inversionistas. 

**Crea un análisis que responda estas preguntas:**

1. **Top 3 canciones más escuchadas** (con nombre de artista)
2. **Género con mejor tasa de completion**
3. **País con más streams totales**
4. **¿Los usuarios Premium escuchan más que los Free?** (streams por usuario)
5. **Artista con mejor crecimiento** entre Enero y Febrero

```
    ╔══════════════════════════════════════════════════════════════════╗
    ║                                                                  ║
    ║   📊 SOUNDWAVE - REPORTE EJECUTIVO Q1 2024                       ║
    ║                                                                  ║
    ║   ┌────────────────────────────────────────────────────────┐    ║
    ║   │ 🎵 TOP 3 CANCIONES                                     │    ║
    ║   │    1. ???????? - ????????                              │    ║
    ║   │    2. ???????? - ????????                              │    ║
    ║   │    3. ???????? - ????????                              │    ║
    ║   └────────────────────────────────────────────────────────┘    ║
    ║                                                                  ║
    ║   ┌────────────────────────────────────────────────────────┐    ║
    ║   │ 📈 MÉTRICAS CLAVE                                      │    ║
    ║   │    • Género líder: ???????? (??% completion)           │    ║
    ║   │    • País líder: ???????? (?? streams)                 │    ║
    ║   │    • Premium vs Free: ??? vs ??? streams/usuario       │    ║
    ║   └────────────────────────────────────────────────────────┘    ║
    ║                                                                  ║
    ║   ┌────────────────────────────────────────────────────────┐    ║
    ║   │ 🚀 ARTISTA DESTACADO                                   │    ║
    ║   │    ???????? - Mayor crecimiento mes a mes              │    ║
    ║   └────────────────────────────────────────────────────────┘    ║
    ║                                                                  ║
    ╚══════════════════════════════════════════════════════════════════╝
```

In [13]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ DESAFÍO FINAL: Genera el reporte ejecutivo completo           ║
# ╚═══════════════════════════════════════════════════════════════╝

print("="*60)
print("📊 SOUNDWAVE - REPORTE EJECUTIVO Q1 2024")
print("="*60)

# 1. Top 3 canciones más escuchadas.
top_3_canciones = (
    streams_completos
    .groupby(['titulo', 'nombre'])
    .size()
    .reset_index(name='streams')
    .sort_values('streams', ascending=False)
    .head(3)
)

print("\n🎵 TOP 3 CANCIONES")
print("-"*40)
for i, fila in enumerate(top_3_canciones.itertuples(index=False), start=1):
    print(f"{i}. {fila.titulo} - {fila.nombre} ({fila.streams} streams)")

# 2. Género con mejor tasa de completion.
completion_genero = (
    streams_completos
    .groupby('genero_principal')['escucha_completa']
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

genero_lider = completion_genero.index[0]
tasa_genero_lider = completion_genero.iloc[0]

print("\n🎸 GÉNERO CON MEJOR ENGAGEMENT")
print("-"*40)
print(f"{genero_lider}: {tasa_genero_lider}% completion")

# 3. País con más streams.
streams_por_pais = (
    streams_usuarios
    .groupby('pais')
    .size()
    .sort_values(ascending=False)
)

pais_lider = streams_por_pais.index[0]
streams_pais_lider = int(streams_por_pais.iloc[0])

print("\n🌎 PAÍS LÍDER EN STREAMS")
print("-"*40)
print(f"{pais_lider}: {streams_pais_lider} streams")

# 4. Comparación Premium vs Free por streams por usuario.
premium_free = (
    streams_usuarios
    .groupby('tipo_cuenta')
    .agg(
        total_streams=('stream_id', 'count'),
        usuarios_unicos=('usuario_id', 'nunique')
    )
)

premium_free['streams_por_usuario'] = (
    premium_free['total_streams'] / premium_free['usuarios_unicos']
).round(2)

print("\n💎 PREMIUM VS FREE (streams por usuario)")
print("-"*40)
print(premium_free[['streams_por_usuario']])

# 5. Artista con mayor crecimiento entre Enero y Febrero.
artista_mes = (
    streams_completos
    .groupby(['nombre', 'mes'])
    .size()
    .reset_index(name='streams')
)

crecimiento_artista = (
    artista_mes
    .pivot(index='nombre', columns='mes', values='streams')
    .fillna(0)
)

crecimiento_artista['crecimiento'] = (
    crecimiento_artista.get('Febrero', 0) - crecimiento_artista.get('Enero', 0)
)

crecimiento_artista = crecimiento_artista.sort_values('crecimiento', ascending=False)
artista_destacado = crecimiento_artista.index[0]
crecimiento_destacado = int(crecimiento_artista.iloc[0]['crecimiento'])

print("\n🚀 ARTISTA DESTACADO (mayor crecimiento)")
print("-"*40)
print(f"{artista_destacado}: {crecimiento_destacado:+d} streams de Enero a Febrero")

print("\n" + "="*60)
print("RESUMEN EJECUTIVO")
print("="*60)
print(f"• Canción #1: {top_3_canciones.iloc[0]['titulo']} - {top_3_canciones.iloc[0]['nombre']}")
print(f"• Género líder: {genero_lider} ({tasa_genero_lider}% completion)")
print(f"• País líder: {pais_lider} ({streams_pais_lider} streams)")
print(f"• Free: {premium_free.loc['Free', 'streams_por_usuario']} streams/usuario")
print(f"• Premium: {premium_free.loc['Premium', 'streams_por_usuario']} streams/usuario")
print(f"• Artista destacado: {artista_destacado} ({crecimiento_destacado:+d} streams)")
print("="*60)


📊 SOUNDWAVE - REPORTE EJECUTIVO Q1 2024

🎵 TOP 3 CANCIONES
----------------------------------------
1. Monaco - Bad Bunny (7 streams)
2. Anti-Hero - Taylor Swift (5 streams)
3. Ella Baila Sola - Peso Pluma (5 streams)

🎸 GÉNERO CON MEJOR ENGAGEMENT
----------------------------------------
R&B: 100.0% completion

🌎 PAÍS LÍDER EN STREAMS
----------------------------------------
México: 14 streams

💎 PREMIUM VS FREE (streams por usuario)
----------------------------------------
             streams_por_usuario
tipo_cuenta                     
Free                        3.17
Premium                     3.10

🚀 ARTISTA DESTACADO (mayor crecimiento)
----------------------------------------
Taylor Swift: +1 streams de Enero a Febrero

RESUMEN EJECUTIVO
• Canción #1: Monaco - Bad Bunny
• Género líder: R&B (100.0% completion)
• País líder: México (14 streams)
• Free: 3.17 streams/usuario
• Premium: 3.1 streams/usuario
• Artista destacado: Taylor Swift (+1 streams)


---

## ✅ Criterios de Evaluación

| Criterio | Puntos | Descripción |
|----------|--------|-------------|
| **Parte 1: concat** | 15 | Combinar correctamente los DataFrames de streams |
| **Parte 2: merge** | 25 | Realizar joins correctos entre tablas |
| **Parte 3: groupby** | 25 | Cálculos agregados correctos |
| **Parte 4: pivot_table** | 20 | Crear reportes en formato matriz |
| **Parte 5: melt** | 5 | (BONUS) Transformar datos correctamente |
| **Desafío Final** | 10 | Análisis completo y correcto |
| **Total** | **100** | |

---

## 🎓 Recursos Adicionales

Si te trabas en algún ejercicio, consulta:

- [Documentación de pd.concat()](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- [Documentación de pd.merge()](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)
- [Documentación de groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- [Documentación de pivot_table()](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)
- [Documentación de melt()](https://pandas.pydata.org/docs/reference/api/pandas.melt.html)

---

**¡Éxito en tu análisis! 🎵📊**